# A FAIR² Mental Health Survey Dataset from Kilifi County, Kenya: Demonstrating AI-Ready Data Standards for Data Infrastructure in Africa
This notebook provides a step-by-step demonstration for exploring the Kilifi County mental health survey dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json

# Define the dataset URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vcs2-05nj/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata  # This is a mlcroissant.datastructure.DatasetMetadata instance

print(f"{metadata.name}: {metadata.description}")

## 2. Data Overview
Review available RecordSets, field IDs, and their associated columns. All IDs are referenced via their `@id` field.

In [ ]:
# Identify all record sets and their IDs
record_sets = []
for rset in metadata.record_sets:
    print(f"RecordSet: {rset.name} | @id: {rset.id}")
    record_sets.append(rset.id)
    print("  Fields and Columns:")
    for field in rset.fields:
        col_ids = [col.id for col in field.columns]
        print(f"    Field: {field.name} | @id: {field.id} | Columns: {col_ids}")
    print("")

# If there are multiple record sets, you can review them above.
record_sets

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. All entities are referenced by their `@id`. Below, an example is shown using the first available record set.

In [ ]:
# We'll extract data for ALL record sets found
dfs = {}
for record_set_id in record_sets:
    records = list(dataset.records(record_set=record_set_id))
    dfs[record_set_id] = pd.DataFrame(records)
    print(f"DataFrame for RecordSet '{record_set_id}' contains {dfs[record_set_id].shape[0]} rows and {dfs[record_set_id].shape[1]} columns.")
    print(f"Columns: {dfs[record_set_id].columns.tolist()[:10]}{'...' if len(dfs[record_set_id].columns)>10 else ''}")
    print(dfs[record_set_id].head(2))
    print("-"*60)

# For continued exploration, select the main record set id
main_record_set_id = record_sets[0]
# Show available columns for the main record set
dfs[main_record_set_id].columns.tolist()

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on survey scores, normalizing numeric fields, and grouping data (e.g., by gender or education level). All fields are referenced using their field or column `@id` as shown above.

In [ ]:
# Let's find a numeric field, e.g., total GAD-7 score, by field ID.
# Please replace the IDs below with the actual @id as printed above for the main record set.
# Example: If the field @id is 'gad7_total_score', you would use that.

all_columns = dfs[main_record_set_id].columns.tolist()

# Try to detect a plausible GAD-7 total score column
possible_gad_cols = [c for c in all_columns if 'gad' in c.lower() and 'score' in c.lower()]
if possible_gad_cols:
    numeric_field_id = possible_gad_cols[0]
else:
    numeric_field_id = all_columns[0]  # fallback

print(f"Selected numeric field for EDA: {numeric_field_id}")

# Filter records with GAD-7 total score > threshold
threshold = 10
df = dfs[main_record_set_id].copy()
df[numeric_field_id] = pd.to_numeric(df[numeric_field_id], errors='coerce')
filtered_df = df[df[numeric_field_id] > threshold]
print(f"Filtered records with {numeric_field_id} > {threshold}: {filtered_df.shape[0]} entries")
print(filtered_df.head())

# Normalize the selected field
filtered_df[f"{numeric_field_id}_normalized"] = (
    (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
)
print(f"Normalized {numeric_field_id} for filtered records:")
print(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

# Try grouping by a categorical field, e.g., gender. Replace with actual field name if needed.
group_field_candidates = [col for col in all_columns if 'gender' in col.lower() or 'sex' in col.lower()]
if group_field_candidates:
    group_field_id = group_field_candidates[0]
    print(f"Grouping by field: {group_field_id}")
    grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean()
    print(f"Mean {numeric_field_id} by {group_field_id} for GAD-7 > {threshold}:")
    print(grouped_df)
else:
    group_field_id = None
    print("No obvious grouping field found.")

## 5. Visualization
Visualize the distribution of a numeric field (e.g., GAD-7 total score) and, if possible, its relationship to categorical data (e.g., gender).


In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Histogram of the total GAD-7 score
plt.figure(figsize=(8, 4))
sns.histplot(df[numeric_field_id].dropna(), kde=True, bins=15, color='royalblue')
plt.title(f"Distribution of {numeric_field_id}")
plt.xlabel(numeric_field_id)
plt.ylabel('Count')
plt.show()

# Boxplot (if grouping field is available)
if group_field_id is not None:
    plt.figure(figsize=(8, 4))
    sns.boxplot(x=group_field_id, y=numeric_field_id, data=df)
    plt.title(f"{numeric_field_id} by {group_field_id}")
    plt.show()

## 6. Conclusion
This notebook demonstrated how to use the `mlcroissant` library to:
- Load and inspect a dataset defined via a Croissant schema URL
- Examine available record sets, fields, and columns by their `@id`
- Extract and manipulate tabular data for exploratory analysis
- Visualize numeric survey metrics such as the GAD-7 total score

Replace detected field and record set IDs as needed based on the dataset's metadata output (Step 2) to customize analyses for your specific needs.